# 06 - Predicting monetary relief

This notebook predicts whether a **closed** complaint's CFPB company response indicates monetary relief. The target is derived from `Company response to consumer`: 1 for the exact `Closed with monetary relief` disposition; 0 for `Closed with non-monetary relief`, `Closed with explanation`, and `Closed without relief`. Missing, in-progress, untimely, bare `Closed`, ambiguous legacy `Closed with relief`, and other unresolved/unrecognized responses are excluded. This is a historical outcome proxy, not a fraud label or a promise about future relief.

When narrative text exists, we compare a structured-only Logistic Regression (regex features + product + mapped ontology category + state) to a model that adds TF-IDF narrative features. Both use `class_weight="balanced"`, a stratified holdout, and training-only stratified cross-validation to tune ROC-AUC. The attached 18.5-million-row CSV has **no narrative column**, so this run detects that and trains only the structured model; it does not fabricate text/regex features or claim a text lift. Reported metrics are test ROC-AUC and precision/recall for the positive monetary-relief class.

In [ ]:
# Import required project tools and check that scikit-learn is installed.
from pathlib import Path
import sys
import json

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.risk_model import print_risk_results, run_risk_model, sample_modeling_rows

input_csv = Path(r"D:\desktop\Downloads\complaints.csv\complaints.csv")
output_dir = project_root / "data" / "processed" / "risk_model"

if not input_csv.is_file():
    raise FileNotFoundError(f"Download the CFPB CSV to {input_csv} before running this notebook.")

# Load only target-source and permitted input columns; do not load company-response flags.
# Scan the 5.3-GB source in bounded passes and retain at most 200,000 rows.
# Sampling preserves the observed relief prevalence instead of balancing labels.
complaints, sampling_manifest = sample_modeling_rows(
    input_csv,
    max_rows=200_000,
    chunksize=250_000,
    random_seed=42,
)
print(json.dumps(sampling_manifest, indent=2))
if not sampling_manifest["narrative_column_available"]:
    print("Narratives are absent: the model comparison will be structured-only.")

In [ ]:
# Construct deterministic regex/ontology fields, then split before model fitting.
# The helper fits imputers, encoders, TF-IDF, and classifiers only within training CV.
risk_results = run_risk_model(
    complaints,
    output_dir=output_dir,
    test_size=0.2,
    cv_folds=3,
    random_seed=42,
    sampling_manifest=sampling_manifest,
)
print_risk_results(risk_results)

In [ ]:
# Compare test ROC-AUC and monetary-relief precision/recall side by side.
comparison = pd.DataFrame(
    [
        {
            "model": model_name.replace("_", " ").title(),
            "best_cv_roc_auc": model_result["best_cv_roc_auc"],
            "test_roc_auc": model_result["test_roc_auc"],
            "relief_precision": model_result["positive_class_precision"],
            "relief_recall": model_result["positive_class_recall"],
            "best_parameters": model_result["best_parameters"],
        }
        for model_name, model_result in risk_results["models"].items()
    ]
)
display(comparison)
print("\nOutcome label definition:")
print(risk_results["metadata"]["target_definition"])
print("\nFields deliberately excluded from predictors:")
print(", ".join(risk_results["metadata"]["removed_from_predictors"]))
if risk_results["metadata"]["text_model_unavailable_reason"]:
    print("\nText comparison unavailable:", risk_results["metadata"]["text_model_unavailable_reason"])

## Leakage controls and interpretation

`Company response to consumer` defines the target, so it is strictly excluded from predictors. `Timely response?` and `Consumer disputed?` are also excluded because they are response/dispute workflow fields that may only be known after complaint handling. The model uses complaint-time narrative, Product, State, and an ontology category mapped from Product/Issue/Sub-issue; regex numeric features are extracted from the narrative. The original Issue and Sub-issue strings are not separately encoded, but contribute to the deterministic category mapping.

The holdout is split before any fitted preprocessing. Imputation, scaling, one-hot encoding, vocabulary/IDF fitting, and parameter selection all happen inside pipelines and cross-validation on training rows. Hyperparameters are chosen by training-only ROC-AUC; the held-out test set is used for final ROC-AUC and positive-class precision/recall at the default 0.5 probability threshold. A low positive-class recall means many relief cases are missed; low precision means more predicted relief cases are false alarms. Inspect class balance and threshold costs before using model scores operationally.

The response-derived target is an outcome, not necessarily a prospective label available at the exact moment of complaint intake. This model is a historical risk-ranking prototype. To forecast future relief, define an explicit prediction timestamp and ensure all inputs were available at that time.